# Stage 2a Benchmark: Vanilla Trendyol vs Trakad-7b-base

**Hedef:** Stage 2a continued pre-training'in **ölçülebilir kazanımını** belgelemek.

**Karşılaştırma:**
1. Vanilla `Trendyol-LLM-7b-base-v1.0` — bizim val set'imizde PPL
2. Bizim `hakansabunis/trakad-7b-base` — aynı val set'te PPL
3. Aynı 5 academic prompt için her iki modelin generations'ı

**Çıktı:** `benchmark_stage2a.json` — paper için sayısal kanıt.

**Runtime:** A100 önerilir (~30-45 dk), T4 da çalışır (~60-80 dk).

**Akış:**
1. Env + Install
2. **RESTART SESSION**
3. HF auth + corpus pull
4. Vanilla Trendyol PPL
5. Trakad-7b-base PPL
6. Side-by-side generations
7. Sonuçları kaydet + HF Hub'a push

## 1. Env + Install (BİR KEZ → RESTART)

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!nvidia-smi
!pip install -q \
  "transformers==4.45.2" "tokenizers>=0.20,<0.21" \
  "datasets>=2.20,<3.0" "accelerate>=0.34,<1.0" \
  sentencepiece

print('\n*** RESTART SESSION NOW ***')

## 2. (Restart sonrası) Env + HF auth

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from getpass import getpass
from huggingface_hub import login
hf_token = getpass('HF write token: ')
login(hf_token, add_to_git_credential=False)

## 3. Pull val set

In [ ]:
from huggingface_hub import hf_hub_download
import pandas as pd
from datasets import Dataset

WORK_DIR = '/content/benchmark'
os.makedirs(WORK_DIR, exist_ok=True)

VAL_PARQUET = hf_hub_download(
    repo_id='hakansabunis/tr-academic-research-agent-index',
    filename='pretrain_corpus/val.parquet',
    repo_type='dataset',
    local_dir=WORK_DIR,
)
val_df = pd.read_parquet(VAL_PARQUET)
print(f'Val chunks: {len(val_df):,}')

val_ds = Dataset.from_pandas(val_df)
def add_labels(ex):
    ex['labels'] = ex['input_ids']
    return ex
val_ds = val_ds.map(add_labels, num_proc=4)

## 4. Vanilla Trendyol-LLM-7b-base PPL

Aynı val set, aynı tokenizer (Trendyol). 14 GB bf16 yüklenir, eval forward pass yapılır.

In [ ]:
import torch, math, time
from transformers import LlamaTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling

BASE = 'Trendyol/Trendyol-LLM-7b-base-v1.0'
print(f'Loading vanilla {BASE}...')
tokenizer = LlamaTokenizer.from_pretrained(BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

vanilla = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=torch.bfloat16, device_map='auto',
)
vanilla.eval()
print(f'  params: {sum(p.numel() for p in vanilla.parameters())/1e9:.2f}B')

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
args = TrainingArguments(
    output_dir='/tmp/eval_vanilla',
    per_device_eval_batch_size=1,
    bf16=True,
    report_to=[],
    dataloader_num_workers=2,
)
trainer = Trainer(model=vanilla, args=args, eval_dataset=val_ds, data_collator=collator)

print('Evaluating vanilla Trendyol on val set (2,228 chunks)...')
t0 = time.time()
vanilla_metrics = trainer.evaluate()
print(f'\nVanilla Trendyol-LLM-7b-base:')
print(f'  val_loss: {vanilla_metrics["eval_loss"]:.4f}')
print(f'  val_PPL:  {math.exp(vanilla_metrics["eval_loss"]):.2f}')
print(f'  eval time: {(time.time()-t0)/60:.1f} min')

VANILLA_LOSS = vanilla_metrics['eval_loss']
VANILLA_PPL = math.exp(VANILLA_LOSS)

# Save vanilla model in CPU for later generation comparison; keep tokenizer
vanilla.to('cpu')
torch.cuda.empty_cache()
import gc; gc.collect()

## 5. Trakad-7b-base PPL (sanity check — Cell 10 zaten 4.88 demişti)

In [ ]:
OURS = 'hakansabunis/trakad-7b-base'
print(f'Loading {OURS}...')
ours = AutoModelForCausalLM.from_pretrained(
    OURS, torch_dtype=torch.bfloat16, device_map='auto',
)
ours.eval()

trainer2 = Trainer(model=ours, args=args, eval_dataset=val_ds, data_collator=collator)

print('Evaluating trakad-7b-base on val set...')
t0 = time.time()
ours_metrics = trainer2.evaluate()
OURS_LOSS = ours_metrics['eval_loss']
OURS_PPL = math.exp(OURS_LOSS)
print(f'\nTrakad-7b-base:')
print(f'  val_loss: {OURS_LOSS:.4f}')
print(f'  val_PPL:  {OURS_PPL:.2f}')
print(f'  eval time: {(time.time()-t0)/60:.1f} min')

## 6. Karşılaştırma tablosu

In [ ]:
rel_improvement = (VANILLA_PPL - OURS_PPL) / VANILLA_PPL * 100
abs_factor = VANILLA_PPL / OURS_PPL

print('=' * 60)
print('STAGE 2a BENCHMARK SONUÇLARI')
print('=' * 60)
print(f'\n{"Model":40s} {"Loss":>8s} {"PPL":>8s}')
print(f'{"Trendyol-LLM-7b-base-v1.0 (vanilla)":40s} {VANILLA_LOSS:>8.4f} {VANILLA_PPL:>8.2f}')
print(f'{"trakad-7b-base (ours)":40s} {OURS_LOSS:>8.4f} {OURS_PPL:>8.2f}')
print(f'\nRelative PPL improvement: {rel_improvement:.1f}%')
print(f'PPL ratio (vanilla / ours): {abs_factor:.2f}x')
print(f'\nVal set: 2,228 chunks of max_seq=2048, packed Turkish academic abstracts')
print(f'Both models evaluated with same tokenizer (Trendyol), same chunks')

## 7. Side-by-side academic generations

Vanilla vs ours — aynı 5 prompt için karşılaştırmalı çıktı.

In [ ]:
# Move ours back to GPU (vanilla is on CPU now)
torch.cuda.empty_cache(); gc.collect()
# 'ours' is already on GPU from previous cell

PROMPTS = [
    'Bu çalışmada, derin öğrenme yöntemleri kullanılarak Türkçe metinler üzerinde',
    'Yenilenebilir enerji kaynaklarından rüzgar enerjisinin Türkiye\'deki potansiyeli',
    'Tıbbi görüntüleme alanında yapay zeka tabanlı tanı sistemleri',
    'Endüstri 4.0 kapsamında Türk imalat sanayisinde dijital dönüşüm uygulamaları',
    'Üniversite öğrencilerinin akademik başarısını etkileyen faktörler',
]

GEN_KWARGS = dict(max_new_tokens=120, do_sample=False, num_beams=1, repetition_penalty=1.05)

samples = []

print('=== TRAKAD-7B-BASE (Ours) generations ===\n')
for p in PROMPTS:
    inputs = tokenizer(p, return_tensors='pt').to(ours.device)
    with torch.no_grad():
        out = ours.generate(**inputs, **GEN_KWARGS)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f'\n→ Prompt: {p}')
    print(f'  Output: {text}\n' + '-'*80)
    samples.append({'prompt': p, 'ours': text, 'vanilla': None})

# Now swap: load vanilla to GPU, evict ours
print('\n\nSwapping models for vanilla generations...')
ours.to('cpu')
torch.cuda.empty_cache(); gc.collect()
vanilla.to('cuda')

print('\n=== VANILLA Trendyol-LLM-7b-base generations ===\n')
for i, p in enumerate(PROMPTS):
    inputs = tokenizer(p, return_tensors='pt').to(vanilla.device)
    with torch.no_grad():
        out = vanilla.generate(**inputs, **GEN_KWARGS)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f'\n→ Prompt: {p}')
    print(f'  Output: {text}\n' + '-'*80)
    samples[i]['vanilla'] = text

## 8. Save benchmark JSON + push to Hub

In [ ]:
import json
BENCHMARK = {
    'eval_set': {
        'source': 'hakansabunis/tr-academic-research-agent-index/pretrain_corpus/val.parquet',
        'n_chunks': len(val_df),
        'max_seq': 2048,
        'domain': 'Turkish academic abstracts (YOK Tez + DergiPark)',
    },
    'vanilla_trendyol': {
        'model_id': 'Trendyol/Trendyol-LLM-7b-base-v1.0',
        'val_loss': round(VANILLA_LOSS, 4),
        'val_ppl': round(VANILLA_PPL, 2),
    },
    'trakad_7b_base': {
        'model_id': 'hakansabunis/trakad-7b-base',
        'val_loss': round(OURS_LOSS, 4),
        'val_ppl': round(OURS_PPL, 2),
    },
    'comparison': {
        'rel_ppl_improvement_percent': round(rel_improvement, 1),
        'ppl_ratio_vanilla_over_ours': round(abs_factor, 2),
    },
    'sample_generations': samples,
    'notes': 'Both models evaluated on identical val chunks (held out from continued pre-training). Generations use greedy decoding (do_sample=False) for reproducibility.',
}

OUT_JSON = f'{WORK_DIR}/benchmark_stage2a.json'
with open(OUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(BENCHMARK, f, ensure_ascii=False, indent=2)
print(f'Saved: {OUT_JSON}')

# Push to HF Hub for reference
from huggingface_hub import HfApi
api = HfApi(token=hf_token)
api.upload_file(
    path_or_fileobj=OUT_JSON,
    path_in_repo='pretrain_corpus/benchmark_stage2a.json',
    repo_id='hakansabunis/tr-academic-research-agent-index',
    repo_type='dataset',
    commit_message='Add Stage 2a benchmark: vanilla Trendyol vs trakad-7b-base PPL + sample generations',
    token=hf_token,
)
print('Pushed benchmark to HF Hub dataset')
print(f'\nFinal summary:\n  Vanilla PPL: {VANILLA_PPL:.2f}')
print(f'  Ours PPL:    {OURS_PPL:.2f}')
print(f'  Improvement: {rel_improvement:.1f}% relative ({abs_factor:.2f}x better)')